In [31]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool
import warnings

warnings.filterwarnings('ignore')

In [32]:
# ==============================================================================
# 0. ハイパーパラメータ設定 (Hyperparameters)
# ==============================================================================
# --- 全体設定 ---
SEED = 42
N_FOLDS_DL = 5      # DLによる特徴抽出のための分割数
N_FOLDS_CB = 10     # CatBoostによる最終学習の分割数 (10回学習)
USE_DUAL_GPU = True # GPUを2枚使うかどうか

# --- Deep Learning設定 (特徴量抽出器) ---
# 時間がかかっても精度を出すため、Transformerを使用
DL_MAX_LEN = 40     # シーケンス長
DL_HIDDEN_DIM = 256 # 隠れ層の次元数（これが抽出される特徴量のサイズになります）
DL_LAYERS = 6       # 層の深さ（深いほど複雑な特徴を捉える）
DL_HEADS = 8        # Attention Head数
DL_EPOCHS = 15      # 学習回数
DL_BATCH_SIZE = 256 # バッチサイズ (GPU2枚なら倍の効率)
DL_LR = 1e-4        # 学習率 (低めでじっくり学習)

# --- CatBoost設定 (メイン学習器) ---
# 精度重視の「重厚な」設定
CB_PARAMS = {
    'iterations': 20000,        # 木の本数。非常に大きく設定し、Early Stoppingで止める
    'learning_rate': 0.005,     # 学習率。0.01以下にすることで学習は遅くなるが精度は向上する
    'depth': 8,                 # 木の深さ。6-10の間。深いほど複雑な関係を学習できる
    'l2_leaf_reg': 5,           # 過学習を防ぐ正則化項
    'loss_function': 'RMSE',
    'random_seed': SEED,
    'task_type': 'GPU',         # GPU使用
    'devices': '0:1',           # Dual GPU指定 (0番と1番)
    'early_stopping_rounds': 500, # 500回改善しなければ停止（我慢強く待つ）
    'verbose': 1000             # ログ出力頻度
}


In [33]:
# ==============================================================================
# 1. Input/Output 定義
# ==============================================================================
# パスの自動判定
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_ensemble.csv'

print(">>> Loading Data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# 基本的な前処理
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
feature_cols = [c for c in train_df.columns if c not in ignore_cols]
train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# 標準化 (DL用)
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print(f"Train: {train_df.shape}, Test: {test_df.shape}")


>>> Loading Data...
Train: (223832, 50), Test: (149511, 49)


In [34]:
# ==============================================================================
# 2. Dataset Class (修正版: ID Padding対応)
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, features, target_col='lever', max_len=40, is_train=True):
        self.df = df
        self.features = features
        self.target_col = target_col
        self.max_len = max_len
        self.is_train = is_train
        self.groups = list(df.groupby('sample_id'))
        self.chunks = []
        for g_idx, (_, group) in enumerate(self.groups):
            for start in range(0, len(group), max_len):
                self.chunks.append((g_idx, start, min(start+max_len, len(group))))

    def __len__(self): return len(self.chunks)

    def __getitem__(self, idx):
        g_idx, start, end = self.chunks[idx]
        _, group = self.groups[g_idx]
        data = group.iloc[start:end]
        
        x_np = data[self.features].values
        actual_len = len(x_np)
        
        # 1. 特徴量のパディング
        x_pad = np.zeros((self.max_len, len(self.features)), dtype=np.float32)
        x_pad[:actual_len] = x_np
        
        # 2. マスクの作成
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:actual_len] = 1.0
        
        # 3. IDのパディング (ここが修正ポイント！)
        # 文字列リストも長さを max_len に揃えないとDataLoaderでエラーになる
        ids_list = data['id'].values.tolist()
        ids_pad = ids_list + ['PAD'] * (self.max_len - actual_len)
        
        ret = {
            'x': torch.tensor(x_pad), 
            'mask': torch.tensor(mask), 
            'ids': ids_pad  # パディング済みのリストを返す
        }
        
        if self.is_train:
            y_np = data[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:actual_len] = y_np
            ret['y'] = torch.tensor(y_pad)
            
        return ret

In [35]:
# ==============================================================================
# 3. DLによる特徴量抽出 (修正版ループ)
# ==============================================================================
print("\n>>> Phase 1: Deep Learning Feature Extraction (Fixed Loop)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(groups)):
    print(f"  DL Fold {fold+1}/{N_FOLDS_DL}")
    
    train_ids = groups[train_idx]
    val_ids = groups[val_idx]
    
    # Dataset再作成 (修正版クラスを使用)
    train_ds = BrainDataset(train_df[train_df['sample_id'].isin(train_ids)], feature_cols)
    val_ds = BrainDataset(train_df[train_df['sample_id'].isin(val_ids)], feature_cols)
    test_ds = BrainDataset(test_df, feature_cols, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=DL_BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=DL_BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=DL_BATCH_SIZE, shuffle=False)
    
    model = FeatureExtractorModel(len(feature_cols), d_model=DL_HIDDEN_DIM, num_layers=DL_LAYERS).to(DEVICE)
    if USE_DUAL_GPU and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=DL_LR)
    criterion = nn.MSELoss(reduction='none')
    
    # --- Training ---
    for epoch in range(DL_EPOCHS):
        model.train()
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            pred, _ = model(x)
            loss = (criterion(pred, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
    
    # --- Feature Extraction ---
    model.eval()
    with torch.no_grad():
        for loader, is_test in [(val_loader, False), (test_loader, True)]:
            for batch in loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].cpu().numpy() # [Batch, Time]
                
                # 特徴量抽出 [Batch, Time, Dim]
                _, features = model(x)
                features = features.cpu().numpy()
                
                # batch['ids'] は [Length, Batch] という形式のタプルのリストになっています
                # 例: batch['ids'][t][b] が、バッチbの時刻tのID文字列
                batch_ids_transposed = batch['ids'] 
                
                # バッチサイズ分ループ
                current_batch_size = x.shape[0]
                
                for b in range(current_batch_size):
                    # このサンプルの有効な長さ
                    valid_len = int(mask[b].sum())
                    
                    # IDの復元 (時刻t=0...valid_len-1 までを取得)
                    real_ids = []
                    for t in range(valid_len):
                        r_id = batch_ids_transposed[t][b]
                        if r_id != 'PAD': # 念のためPADチェック
                            real_ids.append(r_id)
                    
                    # 特徴量の取得
                    real_feats = features[b, :len(real_ids), :]
                    
                    # 辞書に保存
                    for t, r_id in enumerate(real_ids):
                        if is_test:
                            if r_id not in dl_features_test: dl_features_test[r_id] = np.zeros(DL_HIDDEN_DIM)
                            dl_features_test[r_id] += real_feats[t]
                        else:
                            dl_features_train[r_id] = real_feats[t]

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

# Testデータの特徴量を平均化
for k in dl_features_test:
    dl_features_test[k] /= N_FOLDS_DL

print("DL Feature Extraction Completed.")


>>> Phase 1: Deep Learning Feature Extraction (Fixed Loop)...
  DL Fold 1/5
  DL Fold 2/5
  DL Fold 3/5
  DL Fold 4/5
  DL Fold 5/5
DL Feature Extraction Completed.


In [36]:
# ==============================================================================
# 4. データの結合 (Original Features + Deep Features)
# ==============================================================================
print(">>> Merging Features...")

# DL特徴量をDataFrame化
def create_emb_df(feature_dict):
    ids = list(feature_dict.keys())
    # numpy arrayに変換
    matrix = np.stack([feature_dict[i] for i in ids])
    df = pd.DataFrame(matrix, columns=[f'dl_emb_{i}' for i in range(DL_HIDDEN_DIM)])
    df['id'] = ids
    return df

train_emb_df = create_emb_df(dl_features_train)
test_emb_df = create_emb_df(dl_features_test)

# 元データと結合
train_full = train_df.merge(train_emb_df, on='id', how='left')
test_full = test_df.merge(test_emb_df, on='id', how='left')

# NaN処理 (念のため)
train_full = train_full.fillna(0)
test_full = test_full.fillna(0)

# CatBoostで使用する特徴量リスト
use_features = [c for c in train_full.columns if c not in ignore_cols + ['dl_emb_id']] # Embeddingカラムと元カラム

# メモリ解放
del train_df, test_df, dl_features_train, dl_features_test, train_emb_df, test_emb_df
gc.collect()


>>> Merging Features...


0

In [37]:
# ==============================================================================
# 5. CatBoostによる学習 (10-Fold CV)
# ==============================================================================
print("\n>>> Phase 2: CatBoost Training (10-Fold CV)...")

kf_cb = GroupKFold(n_splits=N_FOLDS_CB)
groups = train_full['sample_id']

models = []
oof_preds = np.zeros(len(train_full))
test_preds_accum = np.zeros(len(test_full))

for fold, (train_idx, val_idx) in enumerate(kf_cb.split(train_full, groups=groups)):
    print(f"  CatBoost Fold {fold+1}/{N_FOLDS_CB}")
    
    X_train = train_full.iloc[train_idx][use_features]
    y_train = train_full.iloc[train_idx]['lever']
    X_val = train_full.iloc[val_idx][use_features]
    y_val = train_full.iloc[val_idx]['lever']
    
    # Pool作成
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)
    
    # モデル定義 (Dual GPU指定済み)
    model = CatBoostRegressor(**CB_PARAMS)
    
    # 学習
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    # 予測
    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    
    # Test予測を加算
    test_preds_accum += model.predict(test_full[use_features])
    
    # スコア表示
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"    Fold {fold+1} RMSE: {score:.4f}")
    
    models.append(model)
    
    del X_train, y_train, X_val, y_val, train_pool, val_pool
    gc.collect()




>>> Phase 2: CatBoost Training (10-Fold CV)...
  CatBoost Fold 1/10
0:	learn: 1.3195092	test: 1.2940213	best: 1.2940213 (0)	total: 184ms	remaining: 1h 1m 22s
1000:	learn: 1.0666481	test: 1.0983503	best: 1.0983503 (1000)	total: 20.2s	remaining: 6m 22s
2000:	learn: 1.0290713	test: 1.0865686	best: 1.0865686 (2000)	total: 39.7s	remaining: 5m 56s
3000:	learn: 1.0021923	test: 1.0812643	best: 1.0812549 (2998)	total: 59.1s	remaining: 5m 34s
4000:	learn: 0.9797902	test: 1.0776393	best: 1.0776376 (3999)	total: 1m 17s	remaining: 5m 11s
5000:	learn: 0.9603578	test: 1.0752873	best: 1.0752873 (5000)	total: 1m 36s	remaining: 4m 50s
6000:	learn: 0.9432618	test: 1.0736799	best: 1.0736582 (5988)	total: 1m 56s	remaining: 4m 30s
7000:	learn: 0.9276320	test: 1.0723949	best: 1.0723949 (7000)	total: 2m 15s	remaining: 4m 11s
8000:	learn: 0.9132716	test: 1.0713409	best: 1.0713409 (8000)	total: 2m 35s	remaining: 3m 52s
9000:	learn: 0.9000313	test: 1.0707390	best: 1.0707390 (9000)	total: 2m 54s	remaining: 3m 33

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# 6. 提出ファイルの作成
# ==============================================================================
print("\n>>> Creating Submission File...")

# Test予測の平均化
final_test_preds = test_preds_accum / N_FOLDS_CB
final_test_preds = np.maximum(0, final_test_preds) # 負の値を0に

submission = pd.DataFrame({
    'id': test_full['id'],
    'lever': final_test_preds
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Done! Saved to {SUBMISSION_PATH}")
print(f"Overall CV Score: {np.sqrt(mean_squared_error(train_full['lever'], oof_preds)):.4f}")